# CineForge Colab Video Server — Wan 2.1 T2V 1.3B (realistic, free)

This notebook hosts a **realistic text-to-video model** (Wan 2.1 1.3B, open weights, free) on a free
Colab GPU (T4) and exposes it as a small HTTP API that the CineForge `colab` backend on your laptop
can call.

**How to use**
1. Runtime → Change runtime type → **T4 GPU** (free tier).
2. Run the cells top to bottom. The last cell prints a **public URL** (looks like `https://...proxy...`)
   — it needs no signup, no token, no ngrok.
3. On your laptop, set `COLAB_BASE_URL=<that URL>` in `.env` (or export it), then generate with
   `backend="colab"` in CineForge.
4. Keep this tab open. The URL dies when the Colab runtime disconnects — just re-run the last two cells
   and update `COLAB_BASE_URL`.

First generation takes ~3–5 minutes (model download ~9 GB + warm-up); after that each 5s clip takes
roughly 2–4 minutes on a T4 at 480p.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors fastapi uvicorn nest-asyncio

In [ ]:
import torch
from diffusers import WanPipeline, AutoencoderKLWan
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

print("loading model (first run downloads ~9 GB)...")
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)
pipe.to("cuda")
pipe.enable_vae_tiling()
print("model ready on:", torch.cuda.get_device_name(0))

In [ ]:
import io, os, tempfile, threading, time
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel, Field

app = FastAPI(title="CineForge Colab Video Server")
GEN_LOCK = threading.Lock()   # free tier = one GPU, one generation at a time

# Wan is trained at 832x480 (16:9) / 480x832 (9:16); snap requests onto the grid it knows.
def _snap(v, lo=64, multiple=16):
    return max(lo, (int(v) // multiple) * multiple)

class GenBody(BaseModel):
    prompt: str
    negative_prompt: str = ""
    width: int = Field(default=832, ge=64, le=1280)
    height: int = Field(default=480, ge=64, le=1280)
    fps: int = Field(default=16, ge=5, le=30)
    duration: float = Field(default=5.0, ge=1.0, le=10.0)
    num_inference_steps: int = Field(default=25, ge=4, le=50)
    guidance_scale: float = Field(default=5.0, ge=0.0, le=15.0)
    seed: int | None = None

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "busy": GEN_LOCK.locked()}

@app.post("/generate")
def generate(body: GenBody):
    import random
    negative = body.negative_prompt or (
        "cartoon, anime, illustration, painting, drawing, CGI, 3d render, plastic skin, "
        "blurry, low quality, worst quality, jpeg artifacts, deformed"
    )
    seed = body.seed if body.seed is not None else random.randint(0, 2**31 - 1)
    gen = torch.Generator("cuda").manual_seed(seed)
    num_frames = max(9, min(int(body.duration * body.fps), 121))
    w, h = _snap(body.width), _snap(body.height)

    with GEN_LOCK:
        t0 = time.time()
        out = pipe(
            prompt=body.prompt,
            negative_prompt=negative,
            width=w, height=h, num_frames=num_frames,
            num_inference_steps=body.num_inference_steps,
            guidance_scale=body.guidance_scale,
            generator=gen,
        ).frames[0]
        elapsed = round(time.time() - t0, 1)

    # encode to mp4 in memory and return raw bytes
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        tmp = f.name
    try:
        export_to_video(out, tmp, fps=body.fps)
        data = open(tmp, "rb").read()
    finally:
        os.unlink(tmp)
    return Response(
        content=data,
        media_type="video/mp4",
        headers={
            "X-Seed": str(seed),
            "X-Frames": str(num_frames),
            "X-Elapsed-S": str(elapsed),
            "X-Width": str(w), "X-Height": str(h),
        },
    )

print("API ready: GET /health, POST /generate")

In [ ]:
# sanity check — run one short generation locally before opening it up
r = generate(GenBody(prompt="a photorealistic city street at dusk, people walking, cinematic live-action footage", duration=2.0))
print("sanity generation OK, mp4 bytes:", len(r.body))

In [ ]:
import uvicorn, nest_asyncio
from google.colab.output import eval_js

nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

PUBLIC_URL = eval_js('google.colab.kernel.proxyPort(8000)').strip("/")
print("\n" + "=" * 70)
print("PUBLIC API URL — put this in your laptop's .env as COLAB_BASE_URL:")
print(PUBLIC_URL)
print("=" * 70)

## Optional: quick test from inside Colab

```python
import requests
r = requests.post(f"{PUBLIC_URL}/generate", json={"prompt": "zombies walking down a city street, live-action film, photorealistic", "duration": 5.0}, timeout=1200)
open("test.mp4", "wb").write(r.content)
```

Then on your laptop: add `COLAB_BASE_URL` to `.env` and run CineForge with `--backend colab`.